In [ ]:
# ---------------------------
# ESG_App (Voila-compatible, ipywidgets)
# ---------------------------
%matplotlib inline
import io
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

sns.set_style('whitegrid')

# ---------------------------
# Logging / Debug
# ---------------------------
logger = logging.getLogger("esg_app")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    logger.addHandler(logging.StreamHandler())


def log_debug(message: str):
    logger.info(message)
    with debug_output:
        print(message)


# ---------------------------
# Path helpers (safe for Voila)
# ---------------------------
APP_DIR = Path.cwd()
RULES_FILE_NAME = "investor_rules.json"
RULES_FILE_PATH = APP_DIR / RULES_FILE_NAME


def validate_file_path(file_path, must_exist=False):
    """Return a Path if valid, else None. Prevents None-based file access in Voila."""
    if file_path is None:
        log_debug("[WARN] Received None as file path.")
        return None

    try:
        p = Path(file_path)
    except Exception as exc:
        log_debug(f"[WARN] Invalid file path {file_path!r}: {exc}")
        return None

    if must_exist and not p.exists():
        log_debug(f"[WARN] File does not exist: {p}")
        return None

    return p


# ---------------------------
# ESG Model (minimal)
# ---------------------------
class ESGModel:
    def __init__(self, dataset_path=None):
        self.data = pd.DataFrame()
        safe_path = validate_file_path(dataset_path, must_exist=True) if dataset_path is not None else None
        if safe_path is not None:
            try:
                self.data = pd.read_csv(safe_path)
                log_debug(f"[INFO] Loaded dataset from: {safe_path}")
            except Exception as exc:
                log_debug(f"[WARN] Failed to load dataset {safe_path}: {exc}")


# ---------------------------
# Persistent state
# ---------------------------
if 'esg_state' not in globals():
    esg_state = {
        "logged_in": False,
        "user_type": None,
        "username": "",
        "investor_rules": {}  # sector -> required_years
    }


def load_investor_rules():
    safe_path = validate_file_path(RULES_FILE_PATH, must_exist=False)
    if safe_path is None:
        return {}

    if not safe_path.exists():
        log_debug(f"[INFO] Rules file not found yet at {safe_path}; starting with empty rules.")
        return {}

    try:
        with safe_path.open("r", encoding="utf-8") as f:
            loaded = json.load(f)
        if isinstance(loaded, dict):
            cleaned = {}
            for k, v in loaded.items():
                try:
                    cleaned[str(k)] = int(v)
                except Exception:
                    log_debug(f"[WARN] Skipping invalid rule value for sector {k!r}: {v!r}")
            log_debug(f"[INFO] Loaded {len(cleaned)} investor rule(s) from {safe_path}.")
            return cleaned
        log_debug("[WARN] investor_rules.json is not a dictionary; ignoring contents.")
        return {}
    except Exception as exc:
        log_debug(f"[WARN] Could not read investor rules from {safe_path}: {exc}")
        return {}


def save_investor_rules(rules_dict):
    safe_path = validate_file_path(RULES_FILE_PATH, must_exist=False)
    if safe_path is None:
        return False

    try:
        with safe_path.open("w", encoding="utf-8") as f:
            json.dump(rules_dict, f, indent=2)
        log_debug(f"[INFO] Saved investor rules to {safe_path}.")
        return True
    except Exception as exc:
        log_debug(f"[ERROR] Failed to save investor rules: {exc}")
        return False


# Dedicated debug output so Voila users can inspect issues
debug_output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='6px'))

# initial load
esg_state['investor_rules'] = load_investor_rules()

# ---------------------------
# Widgets: Login / Logout
# ---------------------------
username_widget = widgets.Text(description="Username:")
password_widget = widgets.Password(description="Password:")
user_type_widget = widgets.RadioButtons(options=["Investor", "Company"], description="Login as:")
login_button = widgets.Button(description="Login", button_style='success')
logout_button = widgets.Button(description="Logout", button_style='warning')
main_output = widgets.Output()


def do_logout(_=None):
    esg_state['logged_in'] = False
    esg_state['user_type'] = None
    esg_state['username'] = ""
    with main_output:
        clear_output(wait=True)
        display_login()


logout_button.on_click(do_logout)


def on_login(_):
    with main_output:
        clear_output(wait=True)
        if username_widget.value.strip() and password_widget.value.strip():
            esg_state['logged_in'] = True
            esg_state['user_type'] = user_type_widget.value
            esg_state['username'] = username_widget.value.strip()
            display(HTML(f"<h3>Welcome, {esg_state['username']} ({esg_state['user_type']})</h3>"))
            display(logout_button)
            if esg_state['user_type'] == "Investor":
                show_investor_portal()
            else:
                show_company_portal()
        else:
            display(HTML("<b style='color:red'>Please enter username and password.</b>"))
            display_login()


login_button.on_click(on_login)


def display_login():
    display(HTML("<h3>Login</h3>"))
    display(user_type_widget, username_widget, password_widget, login_button)
    if esg_state['investor_rules']:
        display(HTML("<b>Loaded investor rules:</b>"))
        display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector', 'Required Years']))


# ---------------------------
# Investor portal
# ---------------------------
def show_investor_portal():
    display(HTML("<h4>Investor Portal — Set minimum green years per sector</h4>"))

    main_sectors = [
        "Energy", "Technology", "Finance", "Healthcare", "IT", "Materials",
        "Communication Services", "Consumer Discretionary", "Consumer Staples",
        "Real Estate", "Utilities", "Industrials", "Chemical"
    ]

    slider_widgets = {}
    for s in main_sectors:
        current = int(esg_state['investor_rules'].get(s, 4))
        slider_widgets[s] = widgets.IntSlider(
            value=current, min=1, max=10, step=1, description=s, continuous_update=False
        )
        display(slider_widgets[s])

    saved_rules_out = widgets.Output()
    with saved_rules_out:
        if esg_state['investor_rules']:
            display(HTML("<b>All sectors with saved rules:</b>"))
            display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector', 'Required Years']))
    display(saved_rules_out)

    save_btn = widgets.Button(description="Save Investor Rules", button_style='primary')
    out = widgets.Output()

    def save_rules(_):
        for sector, slider in slider_widgets.items():
            esg_state['investor_rules'][sector] = int(slider.value)

        ok = save_investor_rules(esg_state['investor_rules'])
        with out:
            clear_output(wait=True)
            if ok:
                display(HTML("<b style='color:green'>Investor rules saved.</b>"))
            else:
                display(HTML("<b style='color:red'>Could not save investor rules.</b>"))

            with saved_rules_out:
                clear_output(wait=True)
                display(HTML("<b>All sectors with saved rules:</b>"))
                display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector', 'Required Years']))

    save_btn.on_click(save_rules)
    display(save_btn, out)


# ---------------------------
# Company portal
# ---------------------------
def show_company_portal():
    display(HTML("<h4>Company Portal — Upload CSV to Analyze ESG</h4>"))

    # reload latest persisted rules
    esg_state['investor_rules'] = load_investor_rules() or esg_state['investor_rules']

    if not esg_state['investor_rules']:
        display(HTML("<b style='color:orange'>No investor rules found yet. Please ask an investor to set rules.</b>"))

    upload = widgets.FileUpload(accept='.csv', multiple=False)
    process_btn = widgets.Button(description="Process CSV", button_style='success')
    output = widgets.Output()
    display(upload, process_btn, output)

    def normalize_text(s):
        if s is None:
            return ""
        return str(s).strip().lower()

    def get_required_years_for_sector(sector_text):
        rules = esg_state.get('investor_rules', {})
        if not rules:
            return 4

        if sector_text in rules:
            return int(rules[sector_text])

        sector_norm = normalize_text(sector_text)
        for k, v in rules.items():
            if normalize_text(k) == sector_norm:
                return int(v)

        for k, v in rules.items():
            kn = normalize_text(k)
            if kn and kn in sector_norm:
                return int(v)

        for k, v in rules.items():
            kn = normalize_text(k)
            if sector_norm and sector_norm in kn:
                return int(v)

        return 4

    def extract_uploaded_content(upload_value):
        if upload_value is None:
            log_debug("[WARN] upload.value is None.")
            return None

        if isinstance(upload_value, (tuple, list)):
            if len(upload_value) == 0:
                return None
            uploaded_file = upload_value[0]
            if not isinstance(uploaded_file, dict):
                return None
            return uploaded_file.get('content') or uploaded_file.get('data')

        if isinstance(upload_value, dict):
            if len(upload_value) == 0:
                return None
            uploaded_file = list(upload_value.values())[0]
            if not isinstance(uploaded_file, dict):
                return None
            return uploaded_file.get('content') or uploaded_file.get('data')

        log_debug(f"[WARN] Unsupported upload value type: {type(upload_value)}")
        return None

    def process_uploaded_csv(_):
        with output:
            clear_output(wait=True)
            if not upload.value:
                display(HTML("<b style='color:red'>Please upload a CSV file first.</b>"))
                return

            content = extract_uploaded_content(upload.value)
            if content is None:
                display(HTML("<b style='color:red'>Uploaded file content is missing (None).</b>"))
                return

            try:
                df = pd.read_csv(io.BytesIO(content))
            except Exception as exc:
                display(HTML(f"<b style='color:red'>Error reading CSV: {exc}</b>"))
                return

            required = ['Year', 'Sector']
            for col in required:
                if col not in df.columns:
                    display(HTML(f"<b style='color:red'>CSV must have '{col}' column.</b>"))
                    return
            if 'Ticker' not in df.columns and 'Company' not in df.columns:
                display(HTML("<b style='color:red'>CSV must have 'Ticker' or 'Company' column.</b>"))
                return

            entity_col = 'Ticker' if 'Ticker' in df.columns else 'Company'

            score_candidates = [c for c in df.columns if 'score' in c.lower() and 'esg' in c.lower()]
            if score_candidates:
                df['ESG_Score'] = pd.to_numeric(df[score_candidates[0]], errors='coerce')
            else:
                numeric_cols = [c for c in df.select_dtypes(include=np.number).columns.tolist() if c != 'Year']
                if not numeric_cols:
                    display(HTML("<b style='color:red'>Cannot compute ESG Score: no numeric columns found.</b>"))
                    return
                df['ESG_Score'] = df[numeric_cols].mean(axis=1)

            df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
            df['is_green'] = df['ESG_Score'] >= 50
            df['Sector_norm'] = df['Sector'].astype(str).str.strip()

            grouped = df.groupby(entity_col).agg({
                'Sector_norm': 'first',
                'is_green': 'sum',
                'Year': lambda x: x.nunique(),
                'ESG_Score': 'mean'
            }).rename(columns={
                'Sector_norm': 'Sector',
                'is_green': 'Green_Years',
                'Year': 'Years_On_Record',
                'ESG_Score': 'Avg_ESG_Score'
            }).reset_index()

            esg_state['investor_rules'] = load_investor_rules() or esg_state['investor_rules']
            grouped['Required_Years'] = grouped['Sector'].apply(get_required_years_for_sector)
            grouped['Final_Green_Status'] = grouped.apply(
                lambda row: 'Green' if row['Green_Years'] >= row['Required_Years'] else 'Not Green', axis=1
            )

            display(HTML("<b>Company ESG Report:</b>"))
            report_cols = [entity_col, 'Sector', 'Years_On_Record', 'Green_Years', 'Required_Years', 'Avg_ESG_Score', 'Final_Green_Status']
            display(grouped[report_cols])

            plt.figure(figsize=(8, 4))
            sns.histplot(grouped['Green_Years'], bins=range(0, int(grouped['Green_Years'].max()) + 2), kde=False)
            plt.title("Distribution of Green Years per Company")
            plt.xlabel("Green Years")
            plt.show()

    process_btn.on_click(process_uploaded_csv)


# ---------------------------
# Start UI (must render in Voila)
# ---------------------------
with main_output:
    clear_output(wait=True)
    if esg_state['logged_in']:
        display(HTML(f"<h3>Welcome back, {esg_state['username']} ({esg_state['user_type']})</h3>"))
        display(logout_button)
        if esg_state['user_type'] == "Investor":
            show_investor_portal()
        else:
            show_company_portal()
    else:
        display_login()

# Ensure Voila has a concrete final widget tree to render
ui_root = widgets.VBox([
    main_output,
    widgets.HTML("<hr><b>Debug log</b>"),
    debug_output,
])
display(ui_root)

